In [1]:
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack import get_no_mod, LWEDataset, get_filename_from_params
from ml_attack.utils import get_continuous_reduction_default_params, get_default_params, get_percentage_true_b, get_train_default_params, cmod, clean_secret, check_secret, get_vector_distribution

import numpy as np
import statsmodels.api as sm
from statsmodels.robust.norms import TukeyBiweight


np.set_printoptions(linewidth=np.inf)

In [2]:

filename = "/home/cristian/Documents/ML-attack-on-MLWE/data/data_n_128_k_1_s_binary_7d69d_20.pkl"

if os.path.exists(filename):
    print(f"Loading dataset from {filename}")
    dataset = LWEDataset.load_reduced(filename)
    params = dataset.params
    dataset.initialize_secret()
    dataset.approximate_b()
else:
    print("File not found")

Loading dataset from /home/cristian/Documents/ML-attack-on-MLWE/data/data_n_128_k_1_s_binary_7d69d_20.pkl


In [4]:
_, _, std_B = dataset.get_b_distribution()
choosen_percentage = 0.05

num_selected = int(len(std_B) * choosen_percentage)
selected_indices = np.argsort(std_B)[:num_selected]

A_reduced = dataset.get_A()[selected_indices]
best_b = dataset.best_b[selected_indices]

b_real = get_no_mod(dataset.params, dataset.get_A(), dataset.secret, dataset.get_B())[selected_indices]

exact_candidates = np.sum(best_b == b_real)
total_selection = len(best_b)
print(f"[BEST STD] True B is the best candidate: {exact_candidates} / {total_selection} ({100 * exact_candidates / total_selection:.2f}%)")
# Use statsmodels' RLM with Tukey's biweight loss

for _ in range(100):
    # Fit RLM model
    rlm_model = sm.RLM(best_b, A_reduced, M=TukeyBiweight(c=0.5))
    expected_s, _, std_s = get_vector_distribution(dataset.params, dataset.params['secret_type'], dataset.params['hw'])
    start_params = np.random.normal(loc=expected_s, scale=std_s, size=A_reduced.shape[1])
    rlm_results = rlm_model.fit(maxiter=dataset.params['max_iter'],
                            tol=dataset.params['tol'],
                            start_params=start_params,
                            conv='coefs')

    # Predict using the fitted model
    predicted_b = rlm_results.predict(A_reduced)

    c = np.round((predicted_b - best_b) / dataset.params['q'])
    best_b -= c * dataset.params['q']

    # Calculate matches with b_real
    matches = (best_b == b_real).sum()
    percentage = 100 * matches / len(b_real)
    print(f"[RLM Tukey] Percentage of best_b in b_real: {percentage:.2f}%")


[BEST 100% STD] True B is the best candidate: 1146 / 1879 (60.99%)
[RLM Tukey] Percentage of best_b in b_real: 56.57%
[RLM Tukey] Percentage of best_b in b_real: 56.73%
[RLM Tukey] Percentage of best_b in b_real: 55.77%
[RLM Tukey] Percentage of best_b in b_real: 54.39%
[RLM Tukey] Percentage of best_b in b_real: 54.39%
[RLM Tukey] Percentage of best_b in b_real: 56.52%
[RLM Tukey] Percentage of best_b in b_real: 55.72%
[RLM Tukey] Percentage of best_b in b_real: 54.02%
[RLM Tukey] Percentage of best_b in b_real: 52.74%
[RLM Tukey] Percentage of best_b in b_real: 52.47%
[RLM Tukey] Percentage of best_b in b_real: 53.43%
[RLM Tukey] Percentage of best_b in b_real: 54.82%
[RLM Tukey] Percentage of best_b in b_real: 55.08%
[RLM Tukey] Percentage of best_b in b_real: 55.30%
[RLM Tukey] Percentage of best_b in b_real: 55.51%
[RLM Tukey] Percentage of best_b in b_real: 54.82%
[RLM Tukey] Percentage of best_b in b_real: 53.49%
[RLM Tukey] Percentage of best_b in b_real: 52.85%


KeyboardInterrupt: 

In [3]:
# Use the dataset's best_b_candidates and best_b_probs if available
dataset.approximate_b()
_, _, std_B = dataset.get_b_distribution()
choosen_percentage = 0.05

num_selected = int(len(std_B) * choosen_percentage)
selected_indices = np.argsort(std_B)[:num_selected]

A_reduced = dataset.get_A()[selected_indices]
best_b = dataset.best_b[selected_indices]

b_real = get_no_mod(dataset.params, dataset.get_A(), dataset.secret, dataset.get_B())[selected_indices]

exact_candidates = np.sum(best_b == b_real)
total_selection = len(best_b)
print(f"[BEST STD] True B is the best candidate: {exact_candidates} / {total_selection} ({100 * exact_candidates / total_selection:.2f}%)")
# Use statsmodels' RLM with Tukey's biweight loss

b_candidates = [dataset.b_candidates[i] for i in selected_indices]
b_probs = [dataset.b_probs[i] for i in selected_indices]

n_samples = A_reduced.shape[0]

# --- initialize with expected values of b ---
for em_iter in range(10):
    # === M-step: fit RLM model ===
    expected_s, _, std_s = get_vector_distribution(dataset.params,
                                                  dataset.params['secret_type'],
                                                  dataset.params['hw'])
    start_params = np.random.normal(loc=expected_s,
                                    scale=std_s,
                                    size=A_reduced.shape[1])

    rlm_model = sm.RLM(best_b, A_reduced, M=TukeyBiweight(c=1))
    rlm_results = rlm_model.fit(maxiter=dataset.params['max_iter'],
                                tol=dataset.params['tol'],
                                start_params=start_params,
                                conv='coefs')

    raw_guessed_secret = rlm_results.params

    guessed_secret = clean_secret(raw_guessed_secret, params)
    if check_secret(guessed_secret, dataset.A, dataset.B, params):
        print("Secret guessed correctly!")
        break

    rlm_results.params = guessed_secret

    predicted_b = rlm_results.predict(A_reduced)

    # === E-step: update probabilities for b ===
    new_best_b = []
    for i in range(n_samples):
        candidates = b_candidates[i]
        probs = b_probs[i]

        # likelihood ~ exp(- (candidate - pred)^2)
        likelihoods = np.exp(-0.5 * ((candidates - predicted_b[i])/dataset.params['q'])**2)

        post = probs * likelihoods
        post /= np.sum(post) + 1e-12  # normalize

        # update expectation of b_i
        new_best_b.append(candidates[np.argmax(post)])

        b_probs[i] = post

    new_best_b = np.array(new_best_b)

    # --- check convergence ---
    delta = np.linalg.norm(new_best_b - best_b) / (np.linalg.norm(best_b) + 1e-12)
    best_b = new_best_b

    # Print percentage of real_b in new_best_b
    matches = np.sum(b_real == best_b)
    percentage = 100 * matches / len(b_real)
    print(f"[EM] Percentage of b_real in new_best_b: {percentage:.2f}%")

    if delta < 0.001:
        print(f"Converged at iteration {em_iter}")
        break

[BEST STD] True B is the best candidate: 1146 / 1879 (60.99%)
[EM] Percentage of b_real in new_best_b: 60.40%
[EM] Percentage of b_real in new_best_b: 59.18%


KeyboardInterrupt: 